"""
# Titanic Data Analysis and JSON Export
Author: Louise Plessis
Description: Analyze Titanic passenger data, engineer features, and export to JSON
"""

## Step 1 : Setting Up the Project

In [33]:

import pandas as pd
import numpy as np
import json
from datetime import datetime
from pathlib import Path
 
# Set up paths
DATA_DIR = Path("data")
CSV_FILE = DATA_DIR / "titanic.csv"
JSON_FILE = DATA_DIR / "titanic_data.json"
 
# Create data directory if it doesn't exist
DATA_DIR.mkdir(exist_ok=True)
 
print("Project setup complete!")
print(f"Data directory: {DATA_DIR}")
print(f"CSV file location: {CSV_FILE}")

Project setup complete!
Data directory: data
CSV file location: data/titanic.csv


## Step 2: Importing and Exploring the Data

In [34]:
# Load the CSV into a DataFrame
df = pd.read_csv(CSV_FILE)

print(f"Dataset loaded successfully! Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

Dataset loaded successfully! Shape: (891, 12)

Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

First few rows:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 175

## Step 3: Calculating Descriptive Statistics

In [35]:
# Select numeric columns only
numeric_columns = ['Age', 'Fare']

# Calculate statistics (.mean, median, std)
numeric_stats = df[numeric_columns].agg(['mean', 'median', 'std'])
print(numeric_stats)


              Age       Fare
mean    29.699118  32.204208
median  28.000000  14.454200
std     14.526497  49.693429


## Step 4: Identifying Missing Values

In [36]:
# Count missing values
print("\n" + "="*50)
print("MISSING VALUES ANALYSIS")
print("="*50)
 
missing_data = {}
 
for col in df.columns:
    missing_count = df[col].isnull().sum()
    missing_percent = (missing_count / len(df)) * 100
    missing_data[col] = missing_percent
    print(f"{col}: {missing_count} missing ({missing_percent:.1f}%)")


MISSING VALUES ANALYSIS
PassengerId: 0 missing (0.0%)
Survived: 0 missing (0.0%)
Pclass: 0 missing (0.0%)
Name: 0 missing (0.0%)
Sex: 0 missing (0.0%)
Age: 177 missing (19.9%)
SibSp: 0 missing (0.0%)
Parch: 0 missing (0.0%)
Ticket: 0 missing (0.0%)
Fare: 0 missing (0.0%)
Cabin: 687 missing (77.1%)
Embarked: 2 missing (0.2%)


In [37]:
print("\nColumns with the most missing data:")
for col, percent in sorted(missing_data.items(), key=lambda x: x[1], reverse=True):
    if percent > 0:
        print(f"{col}: {percent:.1f}%")


Columns with the most missing data:
Cabin: 77.1%
Age: 19.9%
Embarked: 0.2%


## Step 5: Feature Engineering 

In [38]:
# Create a copy to work on, so the original df stays untouched
df_features = df.copy()

# Family size
df_features['FamilySize'] = df_features['SibSp'] + df_features['Parch'] + 1
print(df_features[['SibSp', 'Parch', 'FamilySize']].head(10))

# Identify alone passengers
df_features['IsAlone'] = df_features['FamilySize'] == 1
print(df_features[['FamilySize', 'IsAlone']].head(10))

# Define Age groups

def categorize_age(age):
    """Categorize age into groups"""
    if pd.isna(age):
        return 'Unknown'
    elif age < 18:
        return 'Child'
    elif age < 30:
        return 'Young Adult'
    elif age < 50:
        return 'Adult'
    else:
        return 'Senior'

df_features['AgeGroup'] = df_features['Age'].apply(categorize_age)
print(df_features[['Age', 'AgeGroup']].head(10))



   SibSp  Parch  FamilySize
0      1      0           2
1      1      0           2
2      0      0           1
3      1      0           2
4      0      0           1
5      0      0           1
6      0      0           1
7      3      1           5
8      0      2           3
9      1      0           2
   FamilySize  IsAlone
0           2    False
1           2    False
2           1     True
3           2    False
4           1     True
5           1     True
6           1     True
7           5    False
8           3    False
9           2    False
    Age     AgeGroup
0  22.0  Young Adult
1  38.0        Adult
2  26.0  Young Adult
3  35.0        Adult
4  35.0        Adult
5   NaN      Unknown
6  54.0       Senior
7   2.0        Child
8  27.0  Young Adult
9  14.0        Child


In [39]:
# Analyze feature differences between survivors and non-survivors
print("\n" + "="*50)
print("FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED")
print("="*50)
 
# Here is an example to get you started:
print("\nFamily Size by Survival:")
family_survival = df_features.groupby('Survived')['FamilySize'].agg(['mean', 'median', 'std'])
print(family_survival)
 
# Statistical test: Do these features help differentiate?
print("\n" + "="*50)
print("FEATURE DIFFERENTIATION ANALYSIS")
print("="*50)
 
survived = df_features[df_features['Survived'] == 1]
not_survived = df_features[df_features['Survived'] == 0]
 
print("\nFamily Size:")
print(f"  Survived mean: {survived['FamilySize'].mean():.2f}")
print(f"  Not Survived mean: {not_survived['FamilySize'].mean():.2f}")
print(f"  Difference: {abs(survived['FamilySize'].mean() - not_survived['FamilySize'].mean()):.2f}")



FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED

Family Size by Survival:
              mean  median       std
Survived                            
0         1.883424     1.0  1.830669
1         1.938596     2.0  1.186076

FEATURE DIFFERENTIATION ANALYSIS

Family Size:
  Survived mean: 1.94
  Not Survived mean: 1.88
  Difference: 0.06


## Step 6: Creating a Data Export Class

In [40]:
# Create Classes for JSON Export
 
class Passenger:
    """
    Represents a passenger with all their information.
    """
    def __init__(self, passenger_id, name, age, sex, survived, pclass, 
                 fare, embarked=None, family_size=None, is_alone=None, title=None):
        self.passenger_id = int(passenger_id) if pd.notna(passenger_id) else None
        self.name = str(name) if pd.notna(name) else None
        self.age = float(age) if pd.notna(age) else None
        self.sex = str(sex) if pd.notna(sex) else None
        self.survived = int(survived) if pd.notna(survived) else None
        self.pclass = int(pclass) if pd.notna(pclass) else None
        self.fare = float(fare) if pd.notna(fare) else None
        self.embarked = str(embarked) if pd.notna(embarked) else None
        self.family_size = int(family_size) if pd.notna(family_size) else None
        self.is_alone = bool(is_alone) if pd.notna(is_alone) else None
        self.title = str(title) if pd.notna(title) else None
    def to_dict(self):
        """Convert passenger to dictionary for JSON serialization."""
        return {
            'passenger_id': self.passenger_id,
            'name': self.name,
            'age': self.age,
            'sex': self.sex,
            'survived': self.survived,
            'pclass': self.pclass,
            'fare': self.fare,
            'embarked': self.embarked,
            'family_size': self.family_size,
            'is_alone': self.is_alone,
            'title': self.title
        }

 
class TitanicDataset:
    """
    Represents the entire Titanic dataset with methods for JSON export.
    """
    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.passengers = []  # Will store Passenger objects
        self._create_passengers()
    
    def _create_passengers(self):
        """Create Passenger objects from dataframe."""
        for idx, row in self.dataframe.iterrows():
            passenger = Passenger(
                passenger_id = row.get('PassengerId', idx),
                name         = row.get('Name', None),
                age          = row.get('Age', None),
                sex          = row.get('Sex', None),
                survived     = row.get('Survived', None),
                pclass       = row.get('Pclass', None),
                fare         = row.get('Fare', None),
                embarked     = row.get('Embarked', None),
                family_size  = row.get('FamilySize', None),
                is_alone     = row.get('IsAlone', None),
                title        = row.get('Title', None)
            )
            self.passengers.append(passenger)

    
    def to_json(self, filename='titanic_data.json'):
        """Export dataset to JSON file."""
        data = {
            'metadata': {
                'dataset_name': 'Titanic Passenger Dataset',
                'export_date': datetime.now().isoformat(),
                'total_passengers': len(self.passengers),
                'survival_rate': float(self.dataframe['Survived'].mean())
            },
            'passengers': [p.to_dict() for p in self.passengers]
        }

        with open(filename, 'w') as f:
            json.dump(data, f, indent=2)

        print(f"Data exported to {filename}")
        return data
    
    def get_summary_stats(self):
        """Get summary statistics."""
        survived_count = sum(1 for p in self.passengers if p.survived == 1)
        not_survived_count = sum(1 for p in self.passengers if p.survived == 0)

        ages = [p.age for p in self.passengers if p.age is not None]
        fares = [p.fare for p in self.passengers if p.fare is not None]

        return {
            'total_passengers': len(self.passengers),
            'survived': survived_count,
            'did_not_survive': not_survived_count,
            'average_age': sum(ages) / len(ages) if ages else None,
            'average_fare': sum(fares) / len(fares) if fares else None
        }
 
# Create dataset object and export
if 'df_features' in locals() and not df_features.empty:
    # Create a TitanicDataset object
    dataset = TitanicDataset(df_features)

    # Print basic information about the dataset
    print(f"Created dataset with {len(dataset.passengers)} passengers")

    # Get and display summary statistics
    stats = dataset.get_summary_stats()
    print("\nSummary Statistics:")
    for key, value in stats.items():
        print(f"  {key}: {value}")

    # Export to JSON
    dataset.to_json('data/titanic_data.json')
else:
    print("df_features not found. Run the feature engineering step first.")

Created dataset with 891 passengers

Summary Statistics:
  total_passengers: 891
  survived: 342
  did_not_survive: 549
  average_age: 29.69911764705882
  average_fare: 32.204207968574636
Data exported to data/titanic_data.json


## Step 7: Testing and Validation

In [42]:
# Additional validation: Load and inspect JSON
with open(JSON_FILE, 'r', encoding='utf-8') as f:
    json_data = json.load(f)

print("JSON loaded successfully!\n")

# 1. Inspect the top-level structure
print("Top-level keys:", list(json_data.keys()))

# 2. Check the metadata block
print("\nMetadata:")
for key, value in json_data['metadata'].items():
    print(f"  {key}: {value}")

# 3. Verify the number of passenger records
num_passengers = len(json_data['passengers'])
print(f"\nNumber of passenger records: {num_passengers}")
print("Passenger count matches expected (891):", num_passengers == 891)

# 4. Inspect one passenger to confirm all fields are present
print("\nFirst passenger record:")
for key, value in json_data['passengers'][0].items():
    print(f"  {key}: {value}")

# 5. Confirm the expected keys exist
expected_keys = ['metadata', 'passengers']
print("\nAll expected top-level keys present:",
      all(key in json_data for key in expected_keys))

JSON loaded successfully!

Top-level keys: ['metadata', 'passengers']

Metadata:
  dataset_name: Titanic Passenger Dataset
  export_date: 2026-07-07T16:03:04.204937
  total_passengers: 891
  survival_rate: 0.3838383838383838

Number of passenger records: 891
Passenger count matches expected (891): True

First passenger record:
  passenger_id: 1
  name: Braund, Mr. Owen Harris
  age: 22.0
  sex: male
  survived: 0
  pclass: 3
  fare: 7.25
  embarked: S
  family_size: 2
  is_alone: False
  title: None

All expected top-level keys present: True
